<a href="https://colab.research.google.com/github/mahendrapd/RAG-Model/blob/main/aryuveda1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Libraries required to install before executing this model
1. pip install numpy
2. pip install pandas
3. pip install torch
4. pip install PyPDF
5. pip install faiss-gpu
6. pip install tranformers
7. pip install sentence-transformers
8. pip install sentencepiece


In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [9]:
from pypdf import PdfReader

def read_pdf_pypdf(file_path):
    try:
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""  # Handle pages with no text
        return text.strip()
    except FileNotFoundError:
        return "Error: File not found."
    except Exception as e:
        return f"Error reading PDF: {e}"

# Example usage
if __name__ == "__main__":
    pdf_text = read_pdf_pypdf("/content/drive/MyDrive/AryuvedaR/ayutech.pdf")
    #print(pdf_text)

In [20]:
def naive_code_chunk(text, chunk_size=300):
    """Slices plain text without considering code structure."""
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

simple_chunks = naive_code_chunk(pdf_text, chunk_size=300)

In [21]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [22]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [23]:
embeddings = model.encode(
    simple_chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,  # Useful for cosine similarity
    show_progress_bar=True
)

Batches:   0%|          | 0/1011 [00:00<?, ?it/s]

In [24]:
print("Embedding shape:", embeddings.shape)
print("First embedding length:", len(embeddings[0]))

Embedding shape: (32323, 384)
First embedding length: 384


In [25]:
embeddings = embeddings.astype("float32")

In [26]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

In [27]:
index.add(embeddings)

In [28]:
print("Number of vectors in index:", index.ntotal)

Number of vectors in index: 32323


In [29]:
query = "How to reduce sugar level?"

# Embed the query
query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# Search top 3 similar chunks
k = 5
scores, indices = index.search(query_embedding, k)

In [34]:
faiss.write_index(index, "/content/drive/MyDrive/AryuvedaR/chunk_index.faiss")

In [31]:
#index = faiss.read_index("chunk_index.faiss")

In [32]:
retrieved_chunks = [simple_chunks[i] for i in indices[0]]

In [33]:
print("Retrieved Context:\n")
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"{i}. {chunk}\n")

Retrieved Context:

1. s cereals, whole
grains, dals, pulses, sprouts, fruits and vegetables do not raise
blood sugar quickly. Glycaemic index (GI) and glycaemic load
(GL) both should be taken into consideration.
Proteins
The recommended dietary allowance (RDA) for protein is 1 gm/kg
IBW. Pregnant women, lactating women a

2. iabetics need
not restrict carbohydrates. Simply modifying the type of
carbohydrates in the diet will help achieve euglycaemia. Simple
carbohydrates obtained from honey, sugar, jaggery, sugarcane,
jam, glucose, etc. are readily absorbed and are to be avoided
completely. Complex carbohydrates, such a

3. of
consuming minimally processed foods with low glycaemic
index and glycaemic load is recommended in the management
of diabetes. Whole grain products such as whole wheat breads,
brown unpolished rice, oats, and barley tend to produce lower
glycaemic and insulinaemic responses than highly processed
r

4. encourage complex carbohydrates, i.e. mainly
whole grain cerea

In [35]:
context = "\n\n".join(retrieved_chunks)

In [36]:
augmented_prompt = f"""
You are a helpful assistant.

Use the following context to answer the user's question.
If the answer is not contained in the context, say you don't know.

Context:
{context}

Question:
{query}

Answer:
"""

In [37]:
augmented_prompt

"\nYou are a helpful assistant.\n\nUse the following context to answer the user's question.\nIf the answer is not contained in the context, say you don't know.\n\nContext:\ns cereals, whole\ngrains, dals, pulses, sprouts, fruits and vegetables do not raise\nblood sugar quickly. Glycaemic index (GI) and glycaemic load\n(GL) both should be taken into consideration.\nProteins\nThe recommended dietary allowance (RDA) for protein is 1 gm/kg\nIBW. Pregnant women, lactating women a\n\niabetics need\nnot restrict carbohydrates. Simply modifying the type of\ncarbohydrates in the diet will help achieve euglycaemia. Simple\ncarbohydrates obtained from honey, sugar, jaggery, sugarcane,\njam, glucose, etc. are readily absorbed and are to be avoided\ncompletely. Complex carbohydrates, such a\n\nof\nconsuming minimally processed foods with low glycaemic\nindex and glycaemic load is recommended in the management\nof diabetes. Whole grain products such as whole wheat breads,\nbrown unpolished rice, oat

In [38]:
from transformers import pipeline

In [39]:
from huggingface_hub import login

In [40]:
login()

In [41]:
generator = pipeline(
    "text-generation",
    model="google/gemma-2-2b-it"
)

response = generator(
    augmented_prompt,
    max_new_tokens=300,
    do_sample=False
)

print(response[0]["generated_text"])

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



You are a helpful assistant.

Use the following context to answer the user's question.
If the answer is not contained in the context, say you don't know.

Context:
s cereals, whole
grains, dals, pulses, sprouts, fruits and vegetables do not raise
blood sugar quickly. Glycaemic index (GI) and glycaemic load
(GL) both should be taken into consideration.
Proteins
The recommended dietary allowance (RDA) for protein is 1 gm/kg
IBW. Pregnant women, lactating women a

iabetics need
not restrict carbohydrates. Simply modifying the type of
carbohydrates in the diet will help achieve euglycaemia. Simple
carbohydrates obtained from honey, sugar, jaggery, sugarcane,
jam, glucose, etc. are readily absorbed and are to be avoided
completely. Complex carbohydrates, such a

of
consuming minimally processed foods with low glycaemic
index and glycaemic load is recommended in the management
of diabetes. Whole grain products such as whole wheat breads,
brown unpolished rice, oats, and barley tend to produ